# BDC 2026 — OOF Collection 5-Fold: ConvNeXt V2-Base (Versi Laptop)

**Tujuan**: dari nol -- fine-tune ConvNeXt V2-Base dengan `StratifiedKFold` 5-fold di seluruh data train, hasilkan:
1. `oof_predictions_wide.csv` -- probabilitas OOF (setiap gambar train diprediksi oleh model yang TIDAK melihatnya)
2. `test_probs_convnextv2_base.npy` + `test_predictions_convnextv2_base_wide.csv` -- probabilitas test, dari ensemble rata-rata 5 checkpoint fold

**Preprocessing wajib konsisten** (val & test): `SmallestMaxSize` (resize sisi pendek) + `CenterCrop` -- BUKAN `Resize` langsung ke kotak, supaya rasio aspek gambar tidak gepeng.

**Optimasi untuk Laptop (VRAM 4GB)**:
- Batch size dikurangi dari 32 ke 16
- Gradient checkpointing diaktifkan untuk menghemat VRAM
- AMP (Mixed Precision) tetap aktif

File ini adalah dasar untuk notebook selanjutnya: `BDC2026_kNN_ConvNeXt_Adaptation.ipynb`.

In [ ]:
# 1. IMPORT & SEED
import random
import numpy as np
import pandas as pd
from pathlib import Path
from PIL import Image

import torch
import torch.nn as nn
import torch.nn.functional as F
from torch.utils.data import Dataset, DataLoader
from torchvision.transforms import functional as F_tf
import timm
import albumentations as A
from albumentations.pytorch import ToTensorV2

from sklearn.model_selection import StratifiedKFold
from sklearn.metrics import f1_score
from sklearn.utils.class_weight import compute_class_weight

SEED = 42
random.seed(SEED)
np.random.seed(SEED)
torch.manual_seed(SEED)
if torch.cuda.is_available():
    torch.cuda.manual_seed_all(SEED)

DEVICE = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Perangkat: {DEVICE}")
if DEVICE == "cuda":
    print(f"GPU: {torch.cuda.get_device_name(0)}")
    print(f"VRAM: {torch.cuda.get_device_properties(0).total_mem / 1024**3:.1f} GB")

In [ ]:
# 2. PATH & HIPERPARAMETER -- SESUAIKAN dengan struktur folder kamu
# Semua path ABSOLUT -- jangan pakai Path(".") relatif, karena cwd kernel bisa
# bergeser dan bikin output tercecer ke folder "outputs" yang berbeda.
PROJECT_ROOT = Path("C:/Users/MyPC PRO/Downloads/BDC2026")
DATA_DIR = PROJECT_ROOT / "clean_dataset_v3"
OUTPUT_DIR = PROJECT_ROOT / "files (1)" / "outputs"   # kanonik -- hasil lengkap ada di sini

TRAIN_DIR = DATA_DIR / "train"
TEST_DIR = DATA_DIR / "test"
MODEL_DIR = OUTPUT_DIR / "models_convnext_oof"

OUTPUT_DIR.mkdir(parents=True, exist_ok=True)
MODEL_DIR.mkdir(parents=True, exist_ok=True)

CLASSES = ["0_Recyclable", "1_Electronic", "2_Organic"]
LABEL_MAPPING = {c: i for i, c in enumerate(CLASSES)}
CLASS_SHORT = ["recyclable", "electronic", "organic"]
MODEL_KEY = "convnext"

MODEL_NAME = "convnextv2_base"
IMG_SIZE = 224
K_FOLDS = 5
BATCH_SIZE = 64  # RTX 3060 12GB (bukan laptop 4GB) -- naik dari 16
NUM_EPOCHS = 6
NUM_WORKERS = 8  # 32-core CPU tersedia -- naik dari 4 supaya data loading gak jadi bottleneck
HEAD_LR = 5e-4
UNFREEZE_STAGES = 2
USE_GRADIENT_CHECKPOINTING = False  # gak perlu -- VRAM 12GB cukup untuk batch 64 tanpa checkpointing

print("Konfigurasi siap. MODEL_KEY:", MODEL_KEY)
print(f"Batch size: {BATCH_SIZE} (dioptimasi untuk RTX 3060 12GB)")
print(f"Gradient checkpointing: {USE_GRADIENT_CHECKPOINTING}")

In [ ]:
# 3. KATALOG TRAIN & TEST
def build_train_catalog(train_dir: Path):
    records = []
    for label_folder in train_dir.iterdir():
        if label_folder.is_dir() and label_folder.name in LABEL_MAPPING:
            for img_file in label_folder.glob("*"):
                if img_file.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"]:
                    records.append({"filepath": str(img_file), "label": label_folder.name})
    return pd.DataFrame(records)

def build_test_catalog(test_dir: Path):
    records = []
    for img_file in sorted(Path(test_dir).glob("*")):
        if img_file.suffix.lower() in [".jpg", ".jpeg", ".png", ".webp"]:
            records.append({"id": img_file.stem, "filepath": str(img_file)})
    return pd.DataFrame(records)

train_df = build_train_catalog(TRAIN_DIR)
test_df = build_test_catalog(TEST_DIR)

# simpan katalog test
test_df.to_csv(OUTPUT_DIR / "test_catalog.csv", index=False)

print(f"Total gambar train : {len(train_df)} (dokumen: 26.527)")
print(train_df["label"].value_counts())
print(f"\nTotal gambar test  : {len(test_df)} (dokumen: 1.458)")

In [ ]:
# 4. TRANSFORMS -- val/test WAJIB SmallestMaxSize + CenterCrop (bukan Resize kotak langsung)
IMAGENET_MEAN = [0.485, 0.456, 0.406]
IMAGENET_STD = [0.229, 0.224, 0.225]

def build_transforms(img_size: int, train: bool):
    if train:
        return A.Compose([
            A.SmallestMaxSize(max_size=int(img_size * 1.15)),
            A.RandomCrop(height=img_size, width=img_size),
            A.HorizontalFlip(p=0.5),
            A.VerticalFlip(p=0.3),
            A.RandomRotate90(p=0.3),
            A.ColorJitter(brightness=0.2, contrast=0.2, saturation=0.2, hue=0.03, p=0.5),
            A.CoarseDropout(num_holes_range=(1, 6), hole_height_range=(0.02, 0.08),
                             hole_width_range=(0.02, 0.08), p=0.3),
            A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
            ToTensorV2(),
        ])
    return A.Compose([
        A.SmallestMaxSize(max_size=img_size),
        A.CenterCrop(height=img_size, width=img_size),
        A.Normalize(mean=IMAGENET_MEAN, std=IMAGENET_STD),
        ToTensorV2(),
    ])

# Dataset diimpor dari file eksternal (bukan didefinisikan di cell ini) supaya
# NUM_WORKERS>0 tidak hang di Windows -- worker spawn butuh class yang importable.
from dataset_utils_convnextv2_oof import safe_open, TrainValDataset, IndexedDataset, TestDataset

print("Dataset & transform siap.")

In [ ]:
# 5. MODEL FACTORY (fine-tune sebagian -- head + N stage terakhir)
def build_model(model_name, num_classes, device, unfreeze_stages=UNFREEZE_STAGES, use_grad_ckpt=False):
    model = timm.create_model(model_name, pretrained=True, num_classes=num_classes)
    
    # Enable gradient checkpointing jika diminta (hemat VRAM)
    if use_grad_ckpt and hasattr(model, 'gradient_checkpointing_enable'):
        model.gradient_checkpointing_enable()
        print("  Gradient checkpointing diaktifkan")
    elif use_grad_ckpt:
        # Untuk beberapa model, gradient checkpointing bisa diaktifkan via torch
        try:
            from torch.utils.checkpoint import checkpoint
            # Ini akan digunakan di forward pass jika diperlukan
            print("  Menggunakan torch.utils.checkpoint untuk gradient checkpointing")
        except:
            print("  Gradient checkpointing tidak tersedia, melanjutkan tanpa")
    
    for p in model.parameters():
        p.requires_grad = False
    head = model.get_classifier()
    for p in head.parameters():
        p.requires_grad = True
    if hasattr(model, "stages") and unfreeze_stages > 0:
        for s in model.stages[-unfreeze_stages:]:
            for p in s.parameters():
                p.requires_grad = True
    if hasattr(model, "norm_pre"):
        for p in model.norm_pre.parameters():
            p.requires_grad = True
    return model.to(device)

def build_optimizer(model, head_lr, num_epochs):
    head = model.get_classifier()
    head_ids = {id(p) for p in head.parameters()}
    head_params = [p for p in head.parameters() if p.requires_grad]
    backbone_params = [p for p in model.parameters() if p.requires_grad and id(p) not in head_ids]
    groups = [{"params": head_params, "lr": head_lr}]
    if backbone_params:
        groups.append({"params": backbone_params, "lr": head_lr * 0.1})
    optimizer = torch.optim.AdamW(groups, weight_decay=1e-4)
    scheduler = torch.optim.lr_scheduler.CosineAnnealingLR(optimizer, T_max=num_epochs)
    return optimizer, scheduler

print("Model factory siap.")

In [ ]:
# 6. TRAINING 5-FOLD + OOF COLLECTION
num_classes = len(CLASSES)
oof_probs = np.zeros((len(train_df), num_classes))
test_probs_sum = np.zeros((len(test_df), num_classes))

skf = StratifiedKFold(n_splits=K_FOLDS, shuffle=True, random_state=SEED)
y_train_all = train_df["label"].map(LABEL_MAPPING).values
weights = compute_class_weight("balanced", classes=np.arange(num_classes), y=y_train_all)
class_weights = torch.tensor(weights, dtype=torch.float32, device=DEVICE)
criterion = nn.CrossEntropyLoss(weight=class_weights, label_smoothing=0.1)

test_transform = build_transforms(IMG_SIZE, train=False)
test_dataset = TestDataset(test_df, IMG_SIZE, transform=test_transform)
test_loader = DataLoader(test_dataset, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0, prefetch_factor=4 if NUM_WORKERS > 0 else None)

use_amp = DEVICE == "cuda"
scaler = torch.amp.GradScaler("cuda") if use_amp else None

for fold, (train_idx, val_idx) in enumerate(skf.split(train_df, train_df["label"]), start=1):
    print(f"\n===> FOLD {fold}/{K_FOLDS}")
    train_data = TrainValDataset(train_df.iloc[train_idx], LABEL_MAPPING, IMG_SIZE, build_transforms(IMG_SIZE, True))
    val_base = TrainValDataset(train_df.iloc[val_idx], LABEL_MAPPING, IMG_SIZE, build_transforms(IMG_SIZE, False))
    val_data = IndexedDataset(val_base, val_idx.tolist())

    train_loader = DataLoader(train_data, batch_size=BATCH_SIZE, shuffle=True, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0, prefetch_factor=4 if NUM_WORKERS > 0 else None)
    val_loader = DataLoader(val_data, batch_size=BATCH_SIZE * 2, shuffle=False, num_workers=NUM_WORKERS, pin_memory=True, persistent_workers=NUM_WORKERS > 0, prefetch_factor=4 if NUM_WORKERS > 0 else None)

    model = build_model(MODEL_NAME, num_classes, DEVICE, use_grad_ckpt=USE_GRADIENT_CHECKPOINTING)
    optimizer, scheduler = build_optimizer(model, HEAD_LR, NUM_EPOCHS)
    model_path = MODEL_DIR / f"convnextv2_base_fold{fold}.pth"

    best_f1 = -1.0
    for epoch in range(1, NUM_EPOCHS + 1):
        model.train()
        for inputs, labels in train_loader:
            inputs, labels = inputs.to(DEVICE), labels.to(DEVICE)
            optimizer.zero_grad()
            if use_amp:
                with torch.amp.autocast("cuda"):
                    loss = criterion(model(inputs), labels)
                scaler.scale(loss).backward()
                scaler.step(optimizer)
                scaler.update()
            else:
                loss = criterion(model(inputs), labels)
                loss.backward()
                optimizer.step()
        scheduler.step()

        model.eval()
        all_probs, all_ids = [], []
        with torch.no_grad():
            for inputs, labels, batch_ids in val_loader:
                inputs = inputs.to(DEVICE)
                with torch.amp.autocast("cuda", enabled=use_amp):
                    probs = F.softmax(model(inputs), dim=1)
                all_probs.append(probs.float().cpu().numpy())
                all_ids.extend(batch_ids.cpu().tolist() if torch.is_tensor(batch_ids) else batch_ids)

        val_probs = np.concatenate(all_probs, axis=0)
        y_val = y_train_all[all_ids]
        val_f1 = f1_score(y_val, np.argmax(val_probs, axis=1), average="macro")
        print(f"    Epoch {epoch}/{NUM_EPOCHS} -- Val Macro-F1: {val_f1:.4f}")

        if val_f1 > best_f1:
            best_f1 = val_f1
            torch.save(model.state_dict(), model_path)
            oof_probs[all_ids] = val_probs

    print(f"  Fold {fold} selesai -- Best Val Macro-F1: {best_f1:.4f} (checkpoint: {model_path.name})")

    # Prediksi test dengan checkpoint terbaik fold ini
    model.load_state_dict(torch.load(model_path, map_location=DEVICE, weights_only=True))
    model.eval()
    fold_test_probs = []
    with torch.no_grad():
        for inputs, _ in test_loader:
            inputs = inputs.to(DEVICE)
            with torch.amp.autocast("cuda", enabled=use_amp):
                probs = F.softmax(model(inputs), dim=1)
            fold_test_probs.append(probs.float().cpu().numpy())
    test_probs_sum += np.concatenate(fold_test_probs, axis=0) / K_FOLDS

    # Clear cache setelah setiap fold untuk menghemat VRAM
    if DEVICE == "cuda":
        torch.cuda.empty_cache()

overall_oof_f1 = f1_score(y_train_all, np.argmax(oof_probs, axis=1), average="macro")
print(f"\n{'='*60}\nOOF Macro-F1 keseluruhan (5-fold ConvNeXt V2-Base): {overall_oof_f1:.4f}\n{'='*60}")
print("Catatan: test_probs_sum = rata-rata prediksi dari 5 checkpoint fold (bukan model yang dilatih ulang di 100% data).")
print("Ini pendekatan standar & valid untuk stacking; kalau nanti mau retrain di 100% data sebagai model produksi akhir, itu langkah opsional terpisah.")

In [ ]:
# 7. SIMPAN OOF (untuk notebook 2 & 3)
oof_out = train_df[["filepath", "label"]].copy()
oof_out = oof_out.rename(columns={"label": "true_label"})
for i, cls_short in enumerate(CLASS_SHORT):
    oof_out[f"{MODEL_KEY}_prob_{cls_short}"] = oof_probs[:, i]

oof_path = OUTPUT_DIR / "oof_predictions_wide.csv"
oof_out.to_csv(oof_path, index=False)
print(f"[SAVED] {oof_path} -- {len(oof_out)} baris")

# 8. SIMPAN PREDIKSI TEST
np.save(OUTPUT_DIR / "test_probs_convnextv2_base.npy", test_probs_sum)
test_ids_ordered = test_df["id"].tolist()
np.save(OUTPUT_DIR / "test_ids_convnextv2_base.npy", np.array(test_ids_ordered))

test_out = test_df[["id", "filepath"]].copy()
for i, cls_short in enumerate(CLASS_SHORT):
    test_out[f"{MODEL_KEY}_prob_{cls_short}"] = test_probs_sum[:, i]
test_wide_path = OUTPUT_DIR / "test_predictions_convnextv2_base_wide.csv"
test_out.to_csv(test_wide_path, index=False)

print(f"[SAVED] {OUTPUT_DIR / 'test_probs_convnextv2_base.npy'}")
print(f"[SAVED] {test_wide_path} -- {len(test_out)} baris")

## Selesai -- Lanjut ke Notebook Berikutnya

File yang dihasilkan di sini (semua di folder `outputs/`):
- `oof_predictions_wide.csv` -- untuk membangun index k-NN & grid search parameter (Notebook 2)
- `test_predictions_convnextv2_base_wide.csv` + `test_probs_convnextv2_base.npy` -- probabilitas test (Notebook 2 & 3)
- `test_catalog.csv` -- untuk audit visual di Tahap D (Notebook 3)
- `models_convnext_oof/convnextv2_base_fold{1..5}.pth` -- checkpoint tiap fold

**Tips untuk Laptop (VRAM 4GB)**:
- Jika mengalami OOM (Out of Memory), kurangi `BATCH_SIZE` menjadi 8
- Training akan lebih lambat tapi tetap bisa selesai
- Pastikan ventilasi laptop baik untuk mencegah overheating

Lanjut ke: **`BDC2026_kNN_ConvNeXt_Adaptation.ipynb`**